In [1]:
# Importar as bibliotecas para análise de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Carregar a base de dados do Perfect Store
df = pd.read_excel("../data/projeto1_vendas_perfect_store.xlsx")

# Visualizar as cinco primeiras linhas
df.head()

,mes_referencia,codigo_filial,cidade,regiao,score_disponibilidade_cardapio,score_padrao_visual,score_preco_correto,score_execucao_promocional,score_tempo_atendimento,faturamento_mensal,numero_pedidos,ticket_medio
0,2024-01-01,Filial 001,Rio de Janeiro,Sudeste,94.0,82.9,97.2,100.0,78.7,168293.15,3600,46.75
1,2024-01-01,Filial 002,São Paulo,Sudeste,91.3,100.0,87.4,68.5,74.0,133795.86,2202,60.76
2,2024-01-01,Filial 003,Porto Alegre,Sul,67.3,76.6,83.9,83.7,69.3,109384.81,1864,58.68
3,2024-01-01,Filial 004,Curitiba,Sul,80.4,94.0,100.0,85.1,83.7,154228.22,3077,50.12
4,2024-01-01,Filial 005,Curitiba,Sul,80.8,80.6,87.2,100.0,81.8,127114.45,2761,46.04


In [3]:
# Verificar a quantidade de linhas e colunas
print(f"Dimensões: {df.shape[0]} linhas e {df.shape[1]} colunas")

# Verificar os tipos de dados e valores não nulos
df.info()

Dimensões: 600 linhas e 12 colunas
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   mes_referencia                  600 non-null    datetime64[us]
 1   codigo_filial                   600 non-null    str           
 2   cidade                          600 non-null    str           
 3   regiao                          600 non-null    str           
 4   score_disponibilidade_cardapio  600 non-null    float64       
 5   score_padrao_visual             600 non-null    float64       
 6   score_preco_correto             600 non-null    float64       
 7   score_execucao_promocional      600 non-null    float64       
 8   score_tempo_atendimento         600 non-null    float64       
 9   faturamento_mensal              600 non-null    float64       
 10  numero_pedidos                  600 non-null    in

In [4]:
# Verificar registros duplicados
print(f"Registros duplicados: {df.duplicated().sum()}")

# Verificar a quantidade de filiais e meses
print(f"Filiais: {df['codigo_filial'].nunique()}")
print(f"Meses: {df['mes_referencia'].nunique()}")

# Verificar a quantidade de registros por filial
df.groupby("codigo_filial").size().describe()

Registros duplicados: 0
Filiais: 50
Meses: 12


count    50.0
mean     12.0
std       0.0
min      12.0
25%      12.0
50%      12.0
75%      12.0
max      12.0
dtype: float64

In [5]:
# Verificar duplicidade por filial e mês
print("Filial/mês duplicados:",
      df.duplicated(subset=["codigo_filial", "mes_referencia"]).sum())

# Selecionar as colunas de pontuação
score_cols = [col for col in df.columns if col.startswith("score_")]

# Verificar scores fora do intervalo de 0 a 100
print("Scores fora do intervalo:",
      ((df[score_cols] < 0) | (df[score_cols] > 100)).sum().sum())

Filial/mês duplicados: 0
Scores fora do intervalo: 0


## Cálculo do Perfect Store Score (PSS)

O PSS é calculado por meio de uma média ponderada de cinco indicadores operacionais, considerando a importância de cada critério para o desempenho das filiais.

In [6]:
# Definir os pesos dos indicadores operacionais
pesos = {
    "score_disponibilidade_cardapio": 0.30,
    "score_preco_correto": 0.25,
    "score_execucao_promocional": 0.20,
    "score_padrao_visual": 0.15,
    "score_tempo_atendimento": 0.10
}

# Calcular o Perfect Store Score de cada registro
df["pss"] = sum(df[coluna] * peso for coluna, peso in pesos.items())

# Visualizar os resultados
df[["codigo_filial", "mes_referencia", "pss"]].head()

,codigo_filial,mes_referencia,pss
0,Filial 001,2024-01-01,92.805
1,Filial 002,2024-01-01,85.340
2,Filial 003,2024-01-01,76.325
3,Filial 004,2024-01-01,88.610
4,Filial 005,2024-01-01,86.310


In [7]:
# Definir a classificação conforme o PSS
def classificar_pss(pss):
    if pss >= 90:
        return "Loja Perfeita"
    elif pss >= 75:
        return "Atenção"
    else:
        return "Crítico"

# Aplicar a classificação aos registros
df["classificacao_pss"] = df["pss"].apply(classificar_pss)

# Visualizar os resultados
df[["codigo_filial", "mes_referencia", "pss", "classificacao_pss"]].head()

,codigo_filial,mes_referencia,pss,classificacao_pss
0,Filial 001,2024-01-01,92.805,Loja Perfeita
1,Filial 002,2024-01-01,85.340,Atenção
2,Filial 003,2024-01-01,76.325,Atenção
3,Filial 004,2024-01-01,88.610,Atenção
4,Filial 005,2024-01-01,86.310,Atenção


In [8]:
# Contar os registros por classificação
distribuicao_pss = df["classificacao_pss"].value_counts()

# Calcular a distribuição percentual
percentual_pss = df["classificacao_pss"].value_counts(normalize=True) * 100

# Organizar os resultados em uma tabela
resumo_pss = pd.DataFrame({
    "Quantidade": distribuicao_pss,
    "Percentual (%)": percentual_pss.round(2)
})

# Exibir o resumo
resumo_pss

,Quantidade,Percentual (%)
classificacao_pss,,
Atenção,470,78.33
Loja Perfeita,111,18.50
Crítico,19,3.17
